
Data_Cleaning_and_Recommendation

# Data Cleaning


In [1]:

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Projects/Recommender file/final_movie.csv', encoding='latin1')
df.head(5)

df.shape # Main combined dataset with cast,director



Mounted at /content/drive


(2215, 12)

### Importing Libraries


In [3]:
import os

"""### Importing Libraries"""
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib
from scipy import sparse

# NLTK stuff
import nltk

# Downloads (will be quick; needed for tokenization/lemmatization/stopwords)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# make folders to save artifacts
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Cell 2: load your final merged CSV and inspect
print("Rows,Cols:", df.shape)
print("Columns:", df.columns.tolist())
df.head().T  # show first row transposed for quick glance

# Cell 3: auto-detect relevant columns (safe mapping)
def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

title_col = first_existing_col(df, ["title","name","movie_title","original_title"])
overview_col = first_existing_col(df, ["overview","description","plot","summary"])
genres_col = first_existing_col(df, ["genres","genre","genres_x","movie_genres","genres_list"])
cast_col = first_existing_col(df, ["cast","casts","actors","top_cast","cast_list"])
director_col = first_existing_col(df, ["director","directors","crew_director","Director","directed_by"])

print("Mapped columns:")
print(" title:", title_col)
print(" overview:", overview_col)
print(" genres:", genres_col)
print(" cast:", cast_col)
print(" director:", director_col)

# Cell 4: parsing helpers + text cleaner
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def parse_genres(val):
    if pd.isna(val): return ""
    if isinstance(val, list):
        return " ".join([str(x).replace(" ", "_") for x in val])
    s = str(val).strip()
    # if JSON-like list of dicts from TMDb
    if s.startswith("[") or s.startswith("{"):
        try:
            j = json.loads(s)
            # TMDb style: [{"id": 18, "name": "Drama"}, ...]
            if isinstance(j, list):
                names = [it.get("name") if isinstance(it, dict) else str(it) for it in j]
                names = [n for n in names if n]
                return " ".join([n.replace(" ", "_") for n in names])
            if isinstance(j, dict) and "genres" in j:
                return " ".join([g.get("name","").replace(" ", "_") for g in j["genres"]])
        except Exception:
            pass
    # pipe or comma separated
    if "|" in s:
        parts = [p.strip().replace(" ", "_") for p in s.split("|") if p.strip()]
        return " ".join(parts)
    if "," in s:
        parts = [p.strip().replace(" ", "_") for p in s.split(",") if p.strip()]
        return " ".join(parts)
    return s.replace(" ", "_")

def parse_cast(val, top_k=3):
    if pd.isna(val): return ""
    if isinstance(val, list):
        return " ".join([str(x).replace(" ", "_") for x in val[:top_k]])
    s = str(val).strip()
    if s.startswith("["):
        try:
            j = json.loads(s)
            if isinstance(j, list):
                names = []
                for it in j[:top_k]:
                    if isinstance(it, dict):
                        n = it.get("name") or it.get("original_name") or it.get("actor")
                        if n: names.append(n.replace(" ", "_"))
                    else:
                        names.append(str(it).replace(" ", "_"))
                return " ".join(names)
        except Exception:
            pass
    # comma/pipe separated simple cases
    if "," in s:
        parts = [p.strip().replace(" ", "_") for p in s.split(",")[:top_k] if p.strip()]
        return " ".join(parts)
    if "|" in s:
        parts = [p.strip().replace(" ", "_") for p in s.split("|")[:top_k] if p.strip()]
        return " ".join(parts)
    # fallback: return first 3 words joined by underscore
    return "_".join(s.split()[:top_k])

def parse_director(val):
    if pd.isna(val): return ""
    if isinstance(val, list):
        return str(val[0]).replace(" ", "_")
    s = str(val).strip()
    if s.startswith("["):
        try:
            j = json.loads(s)
            # search for crew-like dicts with job Director
            if isinstance(j, list):
                for it in j:
                    if isinstance(it, dict) and (it.get("job","").lower() == "director" or it.get("department","").lower()=="directing"):
                        n = it.get("name")
                        if n: return n.replace(" ", "_")
                # fallback to first name
                if j and isinstance(j[0], dict) and "name" in j[0]:
                    return j[0]["name"].replace(" ", "_")
        except Exception:
            pass
    if "," in s:
        return s.split(",")[0].strip().replace(" ", "_")
    if "|" in s:
        return s.split("|")[0].strip().replace(" ", "_")
    return s.replace(" ", "_")

def clean_text(text):
    if pd.isna(text): return ""
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)            # remove html
    text = re.sub(r"http\S+", " ", text)           # remove urls
    text = re.sub(r"[^a-z0-9_\s]", " ", text)      # keep a-z, digits, underscores and spaces
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

Rows,Cols: (2215, 12)
Columns: ['movieId_x', 'title', 'genres', 'movieId_y', 'release_date', 'overview', 'popularity', 'vote_average', 'vote_count', 'tmdb_id', 'cast', 'director']
Mapped columns:
 title: title
 overview: overview
 genres: genres
 cast: cast
 director: director


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Content Based Filtering with overview


In [5]:
import re
"""## Content Based Filtering with overview"""
nltk.download('punkt_tab')

# Cell 5: build content columns
content_raw = []
content_clean = []

N = len(df)
print(f"Processing {N} rows...")

for i, row in df.iterrows():
    parts = []
    # title (keep original title as is)
    if title_col:
        parts.append(str(row.get(title_col, "")))
    # genres -> parsed
    if genres_col:
        parts.append(parse_genres(row.get(genres_col, "")))
    # cast -> top-3
    if cast_col:
        parts.append(parse_cast(row.get(cast_col, ""), top_k=3))
    # director
    if director_col:
        parts.append(parse_director(row.get(director_col, "")))
    # overview / plot
    if overview_col:
        parts.append(str(row.get(overview_col, "")))

    # also include common extras if present
    for extra in ["keywords", "tags", "plot_keywords"]:
        if extra in df.columns:
            parts.append(str(row.get(extra, "")))

    raw = " ".join([p for p in parts if p])
    clean = clean_text(raw)

    content_raw.append(raw)
    content_clean.append(clean)

    # small progress print every 500 rows
    if (i+1) % 500 == 0:
        print(f"Processed {i+1}/{N} movies")

df["content_raw"] = content_raw
df["content"] = content_clean

# Save cleaned DF
OUT_PATH = "final_movie_cleaned.csv"
df.to_csv(OUT_PATH, index=False)
print("Saved cleaned dataframe to:", OUT_PATH)

# Cell 6 (optional): TF-IDF on `content` and save artifacts
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1,2))
X = tfidf.fit_transform(df["content"].fillna(""))

# save vectorizer + sparse matrix
joblib.dump(tfidf, "/content/drive/MyDrive/Projects/Recommender file/models/tfidf_vectorizer.joblib")
sparse.save_npz("/content/drive/MyDrive/Projects/Recommender file/models/tfidf_matrix.npz", X)

print("TF-IDF vectorizer saved -> models/tfidf_vectorizer.joblib")
print("TF-IDF matrix saved -> models/tfidf_matrix.npz")
print("Matrix shape:", X.shape)

df.sample(1)[["title","content_raw","content"]].T

import joblib
import scipy.sparse as sp

# Load vectorizer
tfidf = joblib.load("/content/drive/MyDrive/Projects/Recommender file/models/tfidf_vectorizer.joblib")

# Load TF-IDF matrix
tfidf_matrix = sp.load_npz("/content/drive/MyDrive/Projects/Recommender file/models/tfidf_matrix.npz")

print("Reloaded TF-IDF matrix:", tfidf_matrix.shape)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Processing 2215 rows...
Processed 500/2215 movies
Processed 1000/2215 movies
Processed 1500/2215 movies
Processed 2000/2215 movies
Saved cleaned dataframe to: final_movie_cleaned.csv
TF-IDF vectorizer saved -> models/tfidf_vectorizer.joblib
TF-IDF matrix saved -> models/tfidf_matrix.npz
Matrix shape: (2215, 20000)
Reloaded TF-IDF matrix: (2215, 20000)


### Step 4b: Cosine Similarity + Recommendation Function


In [6]:
"""### Step 4b: Cosine Similarity + Recommendation Function"""
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity matrix (all movies vs all movies)
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Create a reverse mapping of movie titles to index
indices = pd.Series(df.index, index=df['title']).drop_duplicates()

#     Recommend movies based on content similarity (TF-IDF + Cosine).
def recommend_movies(title, num_recommendations=10):
    # Get index of the movie
    idx = indices[title]

    # Get similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score (highest first, skip itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:num_recommendations+1]

    # Get the indices of recommended movies
    movie_indices = [i[0] for i in sim_scores]

    return df['title'].iloc[movie_indices]



### Example test


In [7]:
"""### Example test"""
print("Recommendations for 'All Dogs Go to Heaven 2 (1996)':")
recommend_movies("All Dogs Go to Heaven 2 (1996)", 10)



Recommendations for 'All Dogs Go to Heaven 2 (1996)':


,title
1632,All Dogs Go to Heaven (1989)
560,James and the Giant Peach (1996)
807,Alice in Wonderland (1951)
521,Pinocchio (1940)
801,"Sword in the Stone, The (1963)"
1625,"Lord of the Rings, The (1978)"
1639,Adventures in Babysitting (1987)
798,Cinderella (1950)
568,Space Jam (1996)
1200,Shiloh (1997)


### Content filtering with Cast,Director and Genre


In [10]:
"""### Content filtering with Cast,Director and Genre"""
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import scipy.sparse as sp
from difflib import get_close_matches

df=pd.read_csv('/content/drive/MyDrive/Projects/Recommender file/final_movie_cleaned.csv')



### 1. Clean the Dataset--


In [11]:
"""### 1. Clean the Dataset--"""
# Drop duplicate rows based on title & director, then reset index
df = df.drop_duplicates(subset=['title', 'director']).reset_index(drop=True)
# Fill NaNs
for col in ['overview', 'cast', 'director', 'genres']:
    df[col] = df[col].fillna('')



### 2. Process Features (Collapses spaces in names)


In [12]:
"""### 2. Process Features (Collapses spaces in names)"""
# "Clint Eastwood" -> "clinteastwood" so we treat names as single unique tokens
df['director_processed'] = df['director'].str.replace(' ', '').str.lower()
df['cast_processed'] = df['cast'].str.replace(' ', '').str.replace(',', ' ').str.lower()
df['genres_processed'] = df['genres'].str.replace('|', ' ', regex=False).str.lower()



### 3. Separate Vectorization (Allows feature weighting)


In [13]:
"""### 3. Separate Vectorization (Allows feature weighting)"""
# Overview (Natural language, standard TF-IDF)
tfidf_overview = TfidfVectorizer(stop_words='english', max_features=5000)
overview_matrix = tfidf_overview.fit_transform(df['overview'])
# Genres
tfidf_genres = TfidfVectorizer()
genres_matrix = tfidf_genres.fit_transform(df['genres_processed'])
# Director
tfidf_director = TfidfVectorizer()
director_matrix = tfidf_director.fit_transform(df['director_processed'])
# Cast
tfidf_cast = TfidfVectorizer(max_features=5000)
cast_matrix = tfidf_cast.fit_transform(df['cast_processed'])



### 4. Save Vectorizers & Matrices (Persisted to models folder on Drive)


In [14]:
"""### 4. Save Vectorizers & Matrices (Persisted to models folder on Drive)"""
joblib.dump(tfidf_overview, "/content/drive/MyDrive/Projects/Recommender file/models/tfidf_overview.joblib")
sp.save_npz("/content/drive/MyDrive/Projects/Recommender file/models/overview_matrix.npz", overview_matrix)
joblib.dump(tfidf_genres, "/content/drive/MyDrive/Projects/Recommender file/models/tfidf_genres.joblib")
sp.save_npz("/content/drive/MyDrive/Projects/Recommender file/models/genres_matrix.npz", genres_matrix)
joblib.dump(tfidf_director, "/content/drive/MyDrive/Projects/Recommender file/models/tfidf_director.joblib")
sp.save_npz("/content/drive/MyDrive/Projects/Recommender file/models/director_matrix.npz", director_matrix)
joblib.dump(tfidf_cast, "/content/drive/MyDrive/Projects/Recommender file/models/tfidf_cast.joblib")
sp.save_npz("/content/drive/MyDrive/Projects/Recommender file/models/cast_matrix.npz", cast_matrix)



### 5. Fix Pandas Series Duplicate Index Bug


In [16]:
"""### 5. Fix Pandas Series Duplicate Index Bug"""
indices = pd.Series(df.index, index=df['title'])
# Keep only the first occurrence of duplicate movie titles in the lookup
indices = indices[~indices.index.duplicated(keep='first')]



### 6. Recommendation Function with Custom Weights

Calculate cosine similarity dynamically for the query movie only.
This is highly efficient and prevents Out-Of-Memory crashes on large datasets.


4. Combine similarities with custom weights
Overview: 30%, Genres: 40%, Director: 15%, Cast: 15%


In [20]:

def recommend_movies(title, num_recommendations=10):
    # 1. Fuzzy matching if exact title not found
    if title not in indices:
        close_matches = get_close_matches(title, df['title'].tolist(), n=1, cutoff=0.6)
        if not close_matches:
            return f"No close match found for '{title}'"
        title = close_matches[0]
        print(f"🔍 Using closest match: {title}")

    # 2. Get the index (this must happen outside the IF block so it runs for exact matches too)
    idx = indices[title]

    # 3. Calculate cosine similarity dynamically for the query movie only
    sim_overview = cosine_similarity(overview_matrix[idx], overview_matrix).flatten()
    sim_genres = cosine_similarity(genres_matrix[idx], genres_matrix).flatten()
    sim_director = cosine_similarity(director_matrix[idx], director_matrix).flatten()
    sim_cast = cosine_similarity(cast_matrix[idx], cast_matrix).flatten()
    combined_sim = (
        0.30 * sim_overview +
        0.40 * sim_genres +
        0.15 * sim_director +
        0.15 * sim_cast
    )

    # 5. Get similarity scores, excluding the query movie itself
    sim_scores = list(enumerate(combined_sim))
    sim_scores = [item for item in sim_scores if item[0] != idx]

    # 6. Sort and slice top-k recommendations
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[:num_recommendations]
    movie_indices = [i[0] for i in sim_scores]

    return df[['title', 'cast', 'director', 'genres']].iloc[movie_indices]

# Test it
recommend_movies("Midnight in the Garden of Good and Evil", 10)



🔍 Using closest match: Midnight in the Garden of Good and Evil (1997)


,title,cast,director,genres
1226,U Turn (1997),"Samantha Ruth Prabhu, Aadhi Pinisetty, Rahul R...",Pawan Kumar,Crime|Drama|Mystery
569,True Crime (1996),"Clint Eastwood, Isaiah Washington, LisaGay Ham...",Clint Eastwood,Mystery|Thriller
132,Clockers (1995),"Harvey Keitel, John Turturro, Delroy Lindo",Spike Lee,Crime|Drama|Mystery
1119,Absolute Power (1997),"Clint Eastwood, Gene Hackman, Ed Harris",Clint Eastwood,Mystery|Thriller
1348,"Spanish Prisoner, The (1997)","Steve Martin, Campbell Scott, Ben Gazzara",David Mamet,Crime|Drama|Mystery|Thriller
1773,Fletch (1985),"Chevy Chase, Tim Matheson, Dana Wheeler-Nicholson",Michael Ritchie,Comedy|Crime|Mystery
442,"Perfect World, A (1993)","Kevin Costner, Clint Eastwood, Laura Dern",Clint Eastwood,Crime|Drama|Thriller
1526,"Negotiator, The (1998)","Samuel L. Jackson, Kevin Spacey, David Morse",F. Gary Gray,Action|Crime|Drama|Mystery|Thriller
533,Primal Fear (1996),"Richard Gere, Laura Linney, Edward Norton",Gregory Hoblit,Crime|Drama|Mystery|Thriller
1932,True Crime (1999),"Clint Eastwood, Isaiah Washington, LisaGay Ham...",Clint Eastwood,Crime|Thriller


Content based Filtering is working all well and good.

## Collaborative Filtering


In [19]:

import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load MovieLens datasets
movies = pd.read_csv("/content/drive/MyDrive/Projects/Recommender file/movies.csv")
ratings = pd.read_csv("/content/drive/MyDrive/Projects/Recommender file/ratings.csv")   # userId, movieId, rating

ratings= ratings.drop('timestamp', axis=1)

# Remove rows (taking standard subset to keep training/testing split fast)
ratings=ratings.iloc[:-50000]

# Save back to CSV
ratings.to_csv("ratings_clean.csv", index=False)

print(ratings.shape, movies.shape)



(50836, 3) (2129, 3)


### item-based CF


In [21]:
"""### item-based CF"""
min_ratings_threshold = 50
movie_rating_counts = ratings['movieId'].value_counts()
popular_movies = movie_rating_counts[movie_rating_counts >= min_ratings_threshold].index

# Keep only ratings for movies that meet the threshold
ratings_filtered = ratings[ratings['movieId'].isin(popular_movies)].copy()

# 2. Calculate user means and center the ratings using the filtered data
user_means = ratings_filtered.groupby('userId')['rating'].transform('mean')
ratings_filtered['rating_centered'] = ratings_filtered['rating'] - user_means

# 3. Create the item-user matrix
item_user_matrix = ratings_filtered.pivot_table(
    index="movieId",
    columns="userId",
    values="rating_centered"
).fillna(0)

# 4. Compute cosine similarity
item_cosine_sim = cosine_similarity(item_user_matrix)
item_sim_df = pd.DataFrame(
    item_cosine_sim,
    index=item_user_matrix.index,
    columns=item_user_matrix.index
)

# 4. Recommendation function with sorting fix
def recommend_similar_movies(movie_title, k=10):
    if movie_title not in movies['title'].values:
        return f"Movie '{movie_title}' not found in dataset!"
    movie_id = movies[movies['title'] == movie_title]['movieId'].values[0]

    # Check if this movie actually has ratings in our matrix
    if movie_id not in item_sim_df.index:
        return f"Movie '{movie_title}' has no ratings in the ratings dataset yet!"
    # Get similarity scores and sort them descending
    scores = item_sim_df[movie_id]
    top_scores = scores.sort_values(ascending=False).iloc[1:k+1]

    # Filter movies, map similarity scores, and sort by them
    recommended = movies[movies['movieId'].isin(top_scores.index)].copy()
    recommended['similarity'] = recommended['movieId'].map(top_scores)

    return recommended.sort_values(by='similarity', ascending=False)

# Test it!
print("\n🎬 Item-based recommendations for Get Shorty (1995):")
recommend_similar_movies("Get Shorty (1995)")




🎬 Item-based recommendations for Get Shorty (1995):


,movieId,title,genres,similarity
156,185,"Net, The (1995)",Action|Crime|Thriller,0.319703
378,434,Cliffhanger (1993),Action|Adventure|Thriller,0.270138
315,357,Four Weddings and a Funeral (1994),Comedy|Romance,0.257175
134,161,Crimson Tide (1995),Drama|Thriller|War,0.175653
197,231,Dumb & Dumber (Dumb and Dumber) (1994),Adventure|Comedy,0.143610
123,150,Apollo 13 (1995),Adventure|Drama|IMAX,0.139377
138,165,Die Hard: With a Vengeance (1995),Action|Crime|Thriller,0.127536
176,208,Waterworld (1995),Action|Adventure|Sci-Fi,0.126030
126,153,Batman Forever (1995),Action|Adventure|Comedy|Crime,0.122200
257,296,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,0.121860


### User Based CF


In [22]:
"""### User Based CF"""
# Create user-item matrix
user_item_matrix = ratings.pivot_table(index="userId", columns="movieId", values="rating").fillna(0)

# Compute cosine similarity (user × user)
user_cosine_sim = cosine_similarity(user_item_matrix)
user_sim_df = pd.DataFrame(user_cosine_sim, index=user_item_matrix.index, columns=user_item_matrix.index)

# Save collaborative models
joblib.dump(item_sim_df, "/content/drive/MyDrive/Projects/Recommender file/models/item_sim_df.joblib")
joblib.dump(user_sim_df, "/content/drive/MyDrive/Projects/Recommender file/models/user_sim_df.joblib")

def recommend_for_user(user_id, k=10, top_n=5):
    if user_id not in user_item_matrix.index:
        return f"User {user_id} not found!"

    # Find most similar users (skip self)
    sim_users = user_sim_df[user_id].sort_values(ascending=False).iloc[1:]

    # Pick top N similar users
    top_users = sim_users.head(top_n).index

    # Movies watched by target user
    target_movies = set(ratings[ratings['userId'] == user_id]['movieId'])

    # Movies watched by top similar users
    similar_user_movies = ratings[ratings['userId'].isin(top_users)]

    # Filter unseen movies
    unseen_movies = similar_user_movies[~similar_user_movies['movieId'].isin(target_movies)]

    if unseen_movies.empty:
        return unseen_movies  # empty DataFrame

    # Sort by avg rating across similar users
    recommendations = (
        unseen_movies.groupby("movieId")["rating"].mean()
        .sort_values(ascending=False)
        .head(k)
        .reset_index()
        .merge(movies, on="movieId")
    )

    return recommendations[["title", "rating"]]

print(recommend_for_user(9, k=10, top_n=5))



                                       title  rating
0                        Pulp Fiction (1994)     5.0
1  Star Wars: Episode IV - A New Hope (1977)     5.0
2           Shawshank Redemption, The (1994)     5.0
3                    Schindler's List (1993)     5.0


**Both collaborative filters are working well independently.**

# Hybrid Recommender

### imports & Fast Feature Extraction Setup


In [24]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Pre-build index mappings for fast NumPy/SciPy operations
print("Pre-building indexes for feature generation...")
movie_id_to_df_idx = {row['movieId_x']: idx for idx, row in df.iterrows()}
movie_id_to_sim_idx = {mid: idx for idx, mid in enumerate(item_sim_df.index)}
item_sim_values = item_sim_df.values

# Pre-build user index mapping for user-based CF
user_id_to_sim_idx = {uid: idx for idx, uid in enumerate(user_sim_df.index)}
user_sim_values = user_sim_df.values

# 2. Pre-calculate global stats for O(1) mathematical adjustments
user_sums = ratings.groupby('userId')['rating'].sum().to_dict()
user_counts = ratings.groupby('userId')['rating'].count().to_dict()

movie_sums = ratings.groupby('movieId')['rating'].sum().to_dict()
movie_counts = ratings.groupby('movieId')['rating'].count().to_dict()

# 3. Create map of user histories (movie -> rating)
user_ratings_map = ratings.groupby('userId').apply(lambda x: dict(zip(x['movieId'], x['rating']))).to_dict()

# Map of movie ratings (user -> rating) for user-based CF lookup
movie_ratings_map = ratings.groupby('movieId').apply(lambda x: dict(zip(x['userId'], x['rating']))).to_dict()

global_avg_rating = ratings['rating'].mean()

def extract_features_for_row(user_id, movie_id, rating):
    # --- User/Movie Baselines (Leakage Adjusted) ---
    u_sum = user_sums.get(user_id, global_avg_rating)
    u_cnt = user_counts.get(user_id, 1)

    m_sum = movie_sums.get(movie_id, global_avg_rating)
    m_cnt = movie_counts.get(movie_id, 1)

    user_avg = (u_sum - rating) / (u_cnt - 1) if u_cnt > 1 else global_avg_rating
    user_cnt = u_cnt - 1

    movie_avg = (m_sum - rating) / (m_cnt - 1) if m_cnt > 1 else global_avg_rating
    movie_cnt = m_cnt - 1

    # Get user rating history excluding current target movie
    user_history = user_ratings_map.get(user_id, {}).copy()
    user_history.pop(movie_id, None)

    # --- 1. Item-Based Collaborative Filtering Feature ---
    collab_item_score = np.nan
    target_sim_idx = movie_id_to_sim_idx.get(movie_id)

    if target_sim_idx is not None and len(user_history) > 0:
        common_mids = [mid for mid in user_history if mid in movie_id_to_sim_idx]
        if common_mids:
            common_sim_indices = [movie_id_to_sim_idx[mid] for mid in common_mids]
            sim_scores = item_sim_values[target_sim_idx, common_sim_indices]
            user_ratings = np.array([user_history[mid] for mid in common_mids])

            weighted_sum = np.dot(sim_scores, user_ratings)
            sum_of_sims = np.abs(sim_scores).sum()

            if sum_of_sims > 0:
                collab_item_score = weighted_sum / sum_of_sims

    # --- 2. User-Based Collaborative Filtering Feature (Leakage Adjusted) ---
    collab_user_score = np.nan
    raters = movie_ratings_map.get(movie_id, {}).copy()
    raters.pop(user_id, None)  # Prevent leakage

    target_user_sim_idx = user_id_to_sim_idx.get(user_id)
    if target_user_sim_idx is not None and raters:
        common_uids = [uid for uid in raters if uid in user_id_to_sim_idx]
        if common_uids:
            common_sim_indices = [user_id_to_sim_idx[uid] for uid in common_uids]
            sim_scores = user_sim_values[target_user_sim_idx, common_sim_indices]
            user_ratings_arr = np.array([raters[uid] for uid in common_uids])

            weighted_sum = np.dot(sim_scores, user_ratings_arr)
            sum_sims = np.abs(sim_scores).sum()

            if sum_sims > 0:
                collab_user_score = weighted_sum / sum_sims

    # --- 3. Content-Based Similarity Feature ---
    content_score = 0.0
    target_df_idx = movie_id_to_df_idx.get(movie_id)

    if target_df_idx is not None and len(user_history) > 0:
        # User liked profile: movies rated >= 4.0 (fallback to all if empty)
        liked_mids = [mid for mid, r in user_history.items() if r >= 4.0]
        if not liked_mids:
            liked_mids = list(user_history.keys())

        profile_df_indices = [movie_id_to_df_idx[mid] for mid in liked_mids if mid in movie_id_to_df_idx]

        if profile_df_indices:
            # Fast sparse dot product logic
            sim_overview = (overview_matrix[target_df_idx] @ overview_matrix[profile_df_indices].T).toarray().flatten()
            sim_genres = (genres_matrix[target_df_idx] @ genres_matrix[profile_df_indices].T).toarray().flatten()
            sim_director = (director_matrix[target_df_idx] @ director_matrix[profile_df_indices].T).toarray().flatten()
            sim_cast = (cast_matrix[target_df_idx] @ cast_matrix[profile_df_indices].T).toarray().flatten()

            combined_sims = (
                0.30 * sim_overview +
                0.40 * sim_genres +
                0.15 * sim_director +
                0.15 * sim_cast
            )
            content_score = float(np.mean(combined_sims))

    return user_avg, user_cnt, movie_avg, movie_cnt, collab_item_score, collab_user_score, content_score

Pre-building indexes for feature generation...


/tmp/ipykernel_4571/2256829638.py:26: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_ratings_map = ratings.groupby('userId').apply(lambda x: dict(zip(x['movieId'], x['rating']))).to_dict()
/tmp/ipykernel_4571/2256829638.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  movie_ratings_map = ratings.groupby('movieId').apply(lambda x: dict(zip(x['userId'], x['rating']))).to_dict()


    Extracts features for a user-movie pair, adjusting counts/sums
    to remove current rating (prevents label leakage during training).
    


###Dataset Generation, Model Training & Validation


In [25]:
"""###Dataset Generation, Model Training & Validation"""
# Create the training dataset
features_list = []
targets = []

# Using full loaded ratings (which is ratings.iloc[:-50000])
sample_ratings = ratings

print(f"Generating features for {len(sample_ratings)} interactions...")
for idx, row in sample_ratings.iterrows():
    u_avg, u_cnt, m_avg, m_cnt, coll_item, coll_user, cont_sc = extract_features_for_row(
        row['userId'], row['movieId'], row['rating']
    )
    features_list.append([u_avg, u_cnt, m_avg, m_cnt, coll_item, coll_user, cont_sc])
    targets.append(row['rating'])

    if (len(features_list)) % 5000 == 0:
        print(f"Processed {len(features_list)}/{len(sample_ratings)} rows")

feature_names = [
    'user_avg_rating', 'user_rating_count',
    'movie_avg_rating', 'movie_rating_count',
    'collab_item_score', 'collab_user_score',
    'content_score'
]
X_df = pd.DataFrame(features_list, columns=feature_names)
y = np.array(targets)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

# Initialize XGBoost Regressor (configured to handle NaN collab_scores gracefully)
model = xgb.XGBRegressor(
    n_estimators=150,
    learning_rate=0.08,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    missing=np.nan,
    random_state=42
)

print("\nTraining XGBoost model...")
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)

# Serialize the trained XGBoost model and feature names to Google Drive
joblib.dump(model, '/content/drive/MyDrive/Projects/Recommender file/models/xgboost_hybrid.joblib')
joblib.dump(feature_names, '/content/drive/MyDrive/Projects/Recommender file/models/feature_names.joblib')
print("\nXGBoost Hybrid model saved -> /content/drive/MyDrive/Projects/Recommender file/models/xgboost_hybrid.joblib")

# Evaluate
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"\n✅ Model Evaluation - RMSE on Test Set: {rmse:.4f}")



Generating features for 50836 interactions...
Processed 5000/50836 rows
Processed 10000/50836 rows
Processed 15000/50836 rows
Processed 20000/50836 rows
Processed 25000/50836 rows
Processed 30000/50836 rows
Processed 35000/50836 rows
Processed 40000/50836 rows
Processed 45000/50836 rows
Processed 50000/50836 rows

Training XGBoost model...
[0]	validation_0-rmse:1.02102
[1]	validation_0-rmse:1.00865
[2]	validation_0-rmse:0.98919
[3]	validation_0-rmse:0.97250
[4]	validation_0-rmse:0.95772
[5]	validation_0-rmse:0.95111
[6]	validation_0-rmse:0.94470
[7]	validation_0-rmse:0.93290
[8]	validation_0-rmse:0.92341
[9]	validation_0-rmse:0.91406
[10]	validation_0-rmse:0.90618
[11]	validation_0-rmse:0.90321
[12]	validation_0-rmse:0.89529
[13]	validation_0-rmse:0.88884
[14]	validation_0-rmse:0.88281
[15]	validation_0-rmse:0.87843
[16]	validation_0-rmse:0.87328
[17]	validation_0-rmse:0.87159
[18]	validation_0-rmse:0.86827
[19]	validation_0-rmse:0.86669
[20]	validation_0-rmse:0.86540
[21]	validation_0

###Machine Learning Hybrid Inference


In [27]:
"""###Machine Learning Hybrid Inference"""
from difflib import get_close_matches
import numpy as np
import pandas as pd
import joblib

def find_title_best(title):
    """Reusing the fuzzy matching logic from your content-based section."""
    if title not in indices:
        close_matches = get_close_matches(title, df['title'].tolist(), n=1, cutoff=0.6)
        if not close_matches:
            return None, None
        title = close_matches[0]
    return title, indices[title]

def get_hybrid_content_scores(idx):
    """Reusing your exact 30/40/15/15 weighting logic for content features."""
    sim_overview = cosine_similarity(overview_matrix[idx], overview_matrix).flatten()
    sim_genres = cosine_similarity(genres_matrix[idx], genres_matrix).flatten()
    sim_director = cosine_similarity(director_matrix[idx], director_matrix).flatten()
    sim_cast = cosine_similarity(cast_matrix[idx], cast_matrix).flatten()

    combined_sim = (0.30 * sim_overview + 0.40 * sim_genres + 0.15 * sim_director + 0.15 * sim_cast)
    return dict(zip(df['movieId_x'], combined_sim))

def get_hybrid_collab_item_scores(user_id):
    """Vectorized calculation of Item-Based Collaborative Filtering scores for all candidate movies."""
    user_history = user_ratings_map.get(user_id, {})
    common_mids = [mid for mid in user_history if mid in movie_id_to_sim_idx]

    if not common_mids:
        return {mid: global_avg_rating for mid in item_sim_df.index}

    common_sim_indices = [movie_id_to_sim_idx[mid] for mid in common_mids]
    user_ratings_arr = np.array([user_history[mid] for mid in common_mids])

    sim_slice = item_sim_values[:, common_sim_indices]
    weighted_sums = np.dot(sim_slice, user_ratings_arr)
    sum_of_sims = np.abs(sim_slice).sum(axis=1)

    preds_array = np.where(sum_of_sims > 0, weighted_sums / sum_of_sims, global_avg_rating)
    preds = dict(zip(item_sim_df.index, preds_array))

    for mid, rating in user_history.items():
        preds[mid] = rating
    return preds

def get_hybrid_collab_user_scores_for_candidates(user_id, candidate_mids):
    """Calculates User-Based CF scores only for the specified candidate movie IDs (highly efficient)."""
    target_user_sim_idx = user_id_to_sim_idx.get(user_id)
    if target_user_sim_idx is None:
        return {mid: global_avg_rating for mid in candidate_mids}

    user_sims = user_sim_values[target_user_sim_idx]
    user_history = user_ratings_map.get(user_id, {})
    preds = {}

    for mid in candidate_mids:
        if mid in user_history:
            preds[mid] = user_history[mid]
            continue

        raters = movie_ratings_map.get(mid, {})
        if not raters:
            preds[mid] = global_avg_rating
            continue

        common_uids = [uid for uid in raters if uid in user_id_to_sim_idx]
        if not common_uids:
            preds[mid] = global_avg_rating
            continue

        common_sim_indices = [user_id_to_sim_idx[uid] for uid in common_uids]
        sims = user_sims[common_sim_indices]
        ratings_arr = np.array([raters[uid] for uid in common_uids])

        weighted_sum = np.dot(sims, ratings_arr)
        sum_sims = np.abs(sims).sum()

        preds[mid] = weighted_sum / sum_sims if sum_sims > 0 else global_avg_rating

    return preds

def hybrid_recommend_xgboost(user_id, movie_title, k=10, load_saved_model=True):
    # 1. Resolve title
    best_title, seed_idx = find_title_best(movie_title)
    if seed_idx is None: return f"Movie '{movie_title}' not found."

    seed_mid = df.iloc[seed_idx]['movieId_x']

    # Load saved model if requested
    if load_saved_model:
        try:
            inference_model = joblib.load('/content/drive/MyDrive/Projects/Recommender file/models/xgboost_hybrid.joblib')
        except Exception:
            inference_model = model
    else:
        inference_model = model

    # 2. Get Scores
    content_scores = get_hybrid_content_scores(seed_idx)
    collab_item_scores = get_hybrid_collab_item_scores(user_id)

    # 3. Candidate Generation (Top 100 from each)
    c_mids = sorted(content_scores, key=content_scores.get, reverse=True)[:100]
    col_mids = sorted(collab_item_scores, key=collab_item_scores.get, reverse=True)[:100]
    candidate_mids = list(set(c_mids) | set(col_mids))

    # Filter history
    user_history = user_ratings_map.get(user_id, {})
    candidate_mids = [m for m in candidate_mids if m != seed_mid and m not in user_history]

    # Calculate User-Based CF scores only for candidate set (extremely fast!)
    collab_user_scores = get_hybrid_collab_user_scores_for_candidates(user_id, candidate_mids)

    # 4. Feature Extraction & XGBoost Prediction
    features = []
    valid_mids = []
    u_avg = user_sums.get(user_id, global_avg_rating) / max(1, user_counts.get(user_id, 0))

    for mid in candidate_mids:
        if mid not in movie_id_to_df_idx: continue
        m_avg = movie_sums.get(mid, global_avg_rating) / max(1, movie_counts.get(mid, 0))
        features.append([
            u_avg,
            user_counts.get(user_id, 0),
            m_avg,
            movie_counts.get(mid, 0),
            collab_item_scores.get(mid, np.nan),
            collab_user_scores.get(mid, np.nan),
            content_scores.get(mid, 0)
        ])
        valid_mids.append(mid)

    X_cand = pd.DataFrame(features, columns=feature_names)
    preds = inference_model.predict(X_cand)

    # 5. Result
    results_df = df[df['movieId_x'].isin(valid_mids)].copy()
    results_df['predicted_rating'] = results_df['movieId_x'].map(dict(zip(valid_mids, preds)))
    results_df['content_similarity'] = results_df['movieId_x'].map(content_scores)
    results_df['item_collab_prediction'] = results_df['movieId_x'].map(collab_item_scores)
    results_df['user_collab_prediction'] = results_df['movieId_x'].map(collab_user_scores)

    return results_df.sort_values(by='predicted_rating', ascending=False).head(k)[
        ['title', 'genres', 'content_similarity', 'item_collab_prediction', 'user_collab_prediction', 'predicted_rating']
    ]

print("\n🎬 Results for User 4 based on Shawshank Redemption, The (1994)")
display(hybrid_recommend_xgboost(4, "Shawshank Redemption(1994)"))


🎬 Results for User 4 based on Shawshank Redemption, The (1994)


,title,genres,content_similarity,item_collab_prediction,user_collab_prediction,predicted_rating
659,"Godfather, The (1972)",Crime|Drama,0.402551,1.111571,4.309184,4.367863
922,"Godfather: Part II, The (1974)",Crime|Drama,0.402862,0.849892,4.151600,4.292921
1734,American History X (1998),Crime|Drama,0.427400,0.640394,4.179229,4.284452
613,Trainspotting (1996),Comedy|Crime|Drama,0.349990,0.653004,4.053307,4.197486
1422,On the Waterfront (1954),Crime|Drama,0.409363,NaN,4.300535,4.094213
1086,Hamlet (1996),Crime|Drama|Romance,0.322240,NaN,4.354706,4.079298
98,Taxi Driver (1976),Crime|Drama|Thriller,0.327817,0.088870,4.022930,4.059299
1883,Office Space (1999),Comedy|Crime,0.294705,NaN,4.018274,4.057505
15,Casino (1995),Crime|Drama,0.404212,NaN,4.115319,4.017350
251,Once Were Warriors (1994),Crime|Drama,0.404068,NaN,4.036457,3.997579


###Model Evaluation


In [28]:
"""###Model Evaluation"""
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



---


In [29]:
# 1. Regression Metrics (Prediction Accuracy)
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("=========================================")
# Print the results
print("📈 REGRESSION METRICS")
print("=========================================")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE):     {mae:.4f}")
print(f"R-squared Score (R2):          {r2:.4f}")
print("=========================================\n")




📈 REGRESSION METRICS
Root Mean Squared Error (RMSE): 0.7326
Mean Absolute Error (MAE):     0.5469
R-squared Score (R2):          0.5082



Calculates Normalized Discounted Cumulative Gain at K.


Calculates what percentage of the top K recommendations the user actually liked (rating >= threshold).


In [30]:
# ----------------------------------------------------------------
# 2. Ranking Metrics (NDCG and Precision)
# ----------------------------------------------------------------
def calculate_ndcg_at_k(actual, predicted, k=5):
    """Calculates Normalized Discounted Cumulative Gain at K."""
    order = np.argsort(predicted)[::-1][:k]
    ordered_actual = np.take(actual, order)

    # Calculate Discounted Cumulative Gain (DCG)
    gains = 2 ** ordered_actual - 1
    discounts = np.log2(np.arange(2, len(ordered_actual) + 2))
    dcg = np.sum(gains / discounts)

    # Calculate Ideal DCG (IDCG)
    ideal_order = np.argsort(actual)[::-1][:k]
    ordered_ideal = np.take(actual, ideal_order)
    ideal_gains = 2 ** ordered_ideal - 1
    ideal_discounts = np.log2(np.arange(2, len(ordered_ideal) + 2))
    idcg = np.sum(ideal_gains / ideal_discounts)

    if idcg == 0:
        return 0.0
    return dcg / idcg

def calculate_precision_at_k(actual, predicted, k=5, threshold=4.0):
    """Calculates what percentage of the top K recommendations the user actually liked (rating >= threshold)."""
    order = np.argsort(predicted)[::-1][:k]
    ordered_actual = np.take(actual, order)
    liked = ordered_actual >= threshold
    return np.mean(liked)


# Map rows in X_test back to their user IDs to group evaluation per user
eval_df = X_test.copy()
eval_df['actual'] = y_test
eval_df['predicted'] = y_pred

# Extract corresponding userIds from the original dataset matching index
eval_df['userId'] = sample_ratings.iloc[X_test.index]['userId'].values

user_ndcgs = []
user_precisions = []

# Group predictions by user and calculate metrics
for uid, group in eval_df.groupby('userId'):
    if len(group) >= 5:  # Only evaluate users with at least 5 test ratings
        act = group['actual'].values
        pred = group['predicted'].values

        user_ndcgs.append(calculate_ndcg_at_k(act, pred, k=5))
        user_precisions.append(calculate_precision_at_k(act, pred, k=5, threshold=4.0))

print("=========================================")
print("🎯 RANKING METRICS (Top-5 recommendations)")
print("=========================================")
print(f"Average NDCG@5:      {np.mean(user_ndcgs):.4f}")
print(f"Average Precision@5: {np.mean(user_precisions):.4f}")
print("=========================================")

🎯 RANKING METRICS (Top-5 recommendations)
Average NDCG@5:      0.8185
Average Precision@5: 0.7863
